# 04 — Filter KQA Pro for reconstruction

Этот ноутбук делает **полуавтоматический отбор** KQA Pro-кандидатов под твой русский multi-hop benchmark.

Что он делает:
1. Загружает `kqapro_complex_candidates.jsonl` из предыдущего этапа.
2. Применяет **жёсткие фильтры** против неподходящих вопросов:
   - `verification` (`Is/Was/Does/...`)
   - `website / url / email / postal code / exact metadata`
   - слишком узкие membership/date microfacts
   - raw `count`, который плохо переводится в list-style
3. Считает **reconstructability score** — насколько вопрос похож на то, что можно естественно пересобрать в русский multi-constraint запрос.
4. Группирует записи по `program_signature`, чтобы не тащить сотни почти одинаковых примеров.
5. Экспортирует:
   - `kqapro_filtered_candidates.*`
   - `kqapro_pattern_bank.*`
   - `kqapro_reconstruction_shortlist.*`
   - `kqapro_manual_review_sheet.csv`

Идея ноутбука:
- **не переводить KQA Pro целиком**,
- а автоматически оставить только **полезные паттерны и кандидаты на реконструкцию**.


In [1]:
%pip install -q pandas pyarrow matplotlib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import json
import math
import re
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd


In [3]:
# =========================
# CONFIG
# =========================

PROJECT_ROOT = Path(".").resolve()

INPUT_DIR = PROJECT_ROOT / "artifacts_kqapro_stage2" / "jsonl"
PREFERRED_INPUT = INPUT_DIR / "kqapro_complex_candidates.jsonl"
FALLBACK_INPUT = INPUT_DIR / "kqapro_balanced_shortlist.jsonl"

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts_kqapro_stage3"
JSONL_DIR = ARTIFACTS_DIR / "jsonl"
CSV_DIR = ARTIFACTS_DIR / "csv"
REPORTS_DIR = ARTIFACTS_DIR / "reports"

for p in [ARTIFACTS_DIR, JSONL_DIR, CSV_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Жёсткие лимиты на итоговый shortlist
MAX_PER_SIGNATURE = 4
MAX_PER_FAMILY = 30
FINAL_SHORTLIST_TARGET = 120

# Порог для авто-классов
KEEP_FOR_RECONSTRUCTION_THRESHOLD = 8
PATTERN_ONLY_THRESHOLD = 4

# Приоритетные семейства reasoning
PREFERRED_FAMILIES = {
    "multi_hop",
    "logical_set",
    "qualifier",
    "temporal",
    "comparison_or_superlative",
}

ALLOWED_BUT_LOWER_PRIORITY = {
    "count",
}

DROP_FAMILIES = {
    "verification",
}

print("PREFERRED_INPUT:", PREFERRED_INPUT)
print("FALLBACK_INPUT :", FALLBACK_INPUT)
print("ARTIFACTS_DIR  :", ARTIFACTS_DIR)


PREFERRED_INPUT: /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage2/jsonl/kqapro_complex_candidates.jsonl
FALLBACK_INPUT : /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage2/jsonl/kqapro_balanced_shortlist.jsonl
ARTIFACTS_DIR  : /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage3


In [4]:
# =========================
# БАЗОВЫЕ УТИЛИТЫ
# =========================

def normalize_spaces(text: Optional[str]) -> str:
    if text is None:
        return ""
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def safe_str(x: Any) -> str:
    if x is None:
        return ""
    return normalize_spaces(str(x))

def maybe_list(x: Any) -> List[Any]:
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]

def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

def save_jsonl(path: Path, rows: Iterable[Dict[str, Any]]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def ensure_list_str(x: Any) -> List[str]:
    if x is None:
        return []
    if isinstance(x, list):
        return [safe_str(v) for v in x if safe_str(v)]
    s = safe_str(x)
    if not s:
        return []
    if s.startswith("[") and s.endswith("]"):
        try:
            data = json.loads(s)
            if isinstance(data, list):
                return [safe_str(v) for v in data if safe_str(v)]
        except Exception:
            pass
    return [s]

def preview_rows(rows: List[Dict[str, Any]], n: int = 3) -> None:
    for i, row in enumerate(rows[:n], start=1):
        print("=" * 120)
        print(f"[row {i}]")
        for k, v in row.items():
            txt = safe_str(v)
            if len(txt) > 400:
                txt = txt[:400] + " ..."
            print(f"{k}: {txt}")


In [5]:
# =========================
# ЗАГРУЗКА ДАННЫХ
# =========================

if PREFERRED_INPUT.exists():
    INPUT_PATH = PREFERRED_INPUT
elif FALLBACK_INPUT.exists():
    INPUT_PATH = FALLBACK_INPUT
else:
    raise FileNotFoundError(
        "Не найден ни kqapro_complex_candidates.jsonl, ни kqapro_balanced_shortlist.jsonl. "
        "Сначала прогони 03_import_kqapro_fixed.ipynb."
    )

rows = read_jsonl(INPUT_PATH)
print("INPUT_PATH:", INPUT_PATH)
print("num_rows  :", len(rows))

assert len(rows) > 0, "Пустой input"

preview_rows(rows, n=2)


INPUT_PATH: /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage2/jsonl/kqapro_complex_candidates.jsonl
num_rows  : 72791
[row 1]
benchmark_id: kqapro_train_000000
source_dataset: kqa_pro
source_split: train
import_mode: template_transfer
role: template_transfer_candidate
question_ru: 
question_en_original: Which town has a TOID of 4000000074573917 and has an OS grid reference of SP8778?
domain: 
reasoning_tags: ['filtering', 'high_compositional_depth', 'logical_set']
primary_reasoning_family: logical_set
hop_count_estimate: 1
program_length: 8
program_functions: ['FindAll', 'FilterStr', 'FilterConcept', 'FindAll', 'FilterStr', 'FilterConcept', 'And', 'What']
program_signature: FindAll -> FilterStr -> FilterConcept -> FindAll -> FilterStr -> FilterConcept -> And -> What
formal_query: SELECT DISTINCT ?e WHERE { ?e <pred:instance_of> ?c . ?c <pred:name> "town" . ?e <TOID> ?pv . ?pv <pred:value> "4000000074573917" . ?e <OS_grid_reference> ?pv_1 . ?pv_1 <pred:value> "SP8778" . }
program: [{

In [6]:
# =========================
# ПРИВЕДЕНИЕ К DATAFRAME
# =========================

df = pd.DataFrame(rows).copy()

# Нормализуем поля, которые нам нужны
for col in [
    "benchmark_id",
    "question_en_original",
    "primary_reasoning_family",
    "program_signature",
    "answer_text",
    "domain",
]:
    if col not in df.columns:
        df[col] = None

df["question_en_original"] = df["question_en_original"].apply(safe_str)
df["primary_reasoning_family"] = df["primary_reasoning_family"].apply(safe_str)
df["program_signature"] = df["program_signature"].apply(safe_str)
df["answer_text"] = df["answer_text"].apply(safe_str)
df["domain"] = df["domain"].apply(safe_str)

if "program_length" not in df.columns:
    df["program_length"] = 0
if "hop_count_estimate" not in df.columns:
    df["hop_count_estimate"] = 0
if "reconstruction_priority_score" not in df.columns:
    df["reconstruction_priority_score"] = 0.0

df["program_length"] = pd.to_numeric(df["program_length"], errors="coerce").fillna(0).astype(int)
df["hop_count_estimate"] = pd.to_numeric(df["hop_count_estimate"], errors="coerce").fillna(0).astype(int)
df["reconstruction_priority_score"] = pd.to_numeric(df["reconstruction_priority_score"], errors="coerce").fillna(0.0)

if "reasoning_tags" not in df.columns:
    df["reasoning_tags"] = [[] for _ in range(len(df))]
df["reasoning_tags"] = df["reasoning_tags"].apply(ensure_list_str)

print("columns:", sorted(df.columns.tolist())[:40], "...")
print("shape  :", df.shape)


columns: ['answer_text', 'benchmark_id', 'choice_count', 'choices', 'domain', 'expected_cardinality', 'formal_query', 'gold_answers', 'gold_qids', 'hop_count_estimate', 'import_mode', 'is_answerable', 'notes', 'primary_reasoning_family', 'program', 'program_functions', 'program_length', 'program_signature', 'question_en_original', 'question_ru', 'reasoning_tags', 'reconstruction_priority_score', 'reconstruction_prompt', 'role', 'source_dataset', 'source_split'] ...
shape  : (72791, 26)


## Жёсткие фильтры

Ниже задаются **эвристики отсева**. Они специально ориентированы на твою постановку:

- нужны **сложные compositional паттерны**;
- нужен потенциал для **русской reconstruction в list-style / multi-constraint формат**;
- не нужны вопросы типа `Is ...?`, `Was ...?`, `Does ...?`, `official website`, `postal code`, `email`, URL и другие технические KB-факты.


In [7]:
# =========================
# ЭВРИСТИКИ ОТСЕВА И SCORE
# =========================

BINARY_PREFIXES = ("is ", "was ", "were ", "are ", "does ", "did ", "do ", "has ", "have ", "had ", "can ")
COUNT_PREFIXES = ("how many ", "how much ", "what number ")
META_KEYWORDS = [
    "official website", "website", "web site", "url", "email", "postal code",
    "iconclass", "isni", "isbn", "issn", "catalog code", "identifier",
    "phone number", "fax", "twitter", "instagram", "facebook", "homepage",
    "mass not equal", "notation equal", "equal to", "not equal to",
]
TECHNICAL_LOCATION_KEYWORDS = [
    "postal code", "address", "located in the postal code", "email address",
]
DATE_MICROFACT_KEYWORDS = [
    "beginning in", "ending in", "start time", "end time", "date of birth",
    "born after", "born before", "dissolved after", "founded before", "founded after",
    "until ", "since ", "after 19", "before 19", "after 20", "before 20",
]
MEMBERSHIP_KEYWORDS = [
    "member of", "cast member", "team member", "association football club", "club that has",
    "voice actor", "spouse of", "partnered with", "received by", "awarded to",
]

def q_lower(text: str) -> str:
    return safe_str(text).lower()

def has_url(text: str) -> bool:
    t = q_lower(text)
    return ("http://" in t) or ("https://" in t) or ("www." in t)

def has_email(text: str) -> bool:
    t = q_lower(text)
    return bool(re.search(r"\b\S+@\S+\.\S+\b", t))

def starts_binary(text: str) -> bool:
    t = q_lower(text)
    return any(t.startswith(p) for p in BINARY_PREFIXES)

def starts_count(text: str) -> bool:
    t = q_lower(text)
    return any(t.startswith(p) for p in COUNT_PREFIXES)

def contains_meta_keyword(text: str) -> bool:
    t = q_lower(text)
    return any(k in t for k in META_KEYWORDS)

def contains_technical_location(text: str) -> bool:
    t = q_lower(text)
    return any(k in t for k in TECHNICAL_LOCATION_KEYWORDS)

def contains_date_microfact(text: str) -> bool:
    t = q_lower(text)
    return any(k in t for k in DATE_MICROFACT_KEYWORDS)

def contains_membership_microfact(text: str) -> bool:
    t = q_lower(text)
    return any(k in t for k in MEMBERSHIP_KEYWORDS)

def parenthesis_count(text: str) -> int:
    t = safe_str(text)
    return t.count("(") + t.count(")")

def long_and_nested(text: str) -> bool:
    t = safe_str(text)
    return (len(t) > 220) or (parenthesis_count(t) >= 4)

def list_style_friendly(text: str) -> bool:
    t = q_lower(text)
    if starts_binary(t):
        return False
    if has_url(t) or has_email(t):
        return False
    if "official website" in t or "postal code" in t or "email address" in t:
        return False
    if starts_count(t):
        return False
    # Более пригодные к list-style вопросы обычно начинаются так:
    good_starts = (
        "which ", "what ", "who ", "where ", "when ", "name ", "regarding ",
        "through ", "from which ", "what city ", "what country ", "what state ",
    )
    return any(t.startswith(p) for p in good_starts)

def program_has_set_ops(signature: str) -> bool:
    s = safe_str(signature).lower()
    tokens = ["and", "or", "selectbetween", "selectamong", "filterconcept", "filterstr", "qfilter"]
    return any(tok in s for tok in tokens)

def hard_drop_reasons(row: pd.Series) -> List[str]:
    q = row["question_en_original"]
    fam = row["primary_reasoning_family"]

    reasons = []

    if fam in DROP_FAMILIES:
        reasons.append("family=verification")

    if starts_binary(q):
        reasons.append("binary_question")

    if has_url(q):
        reasons.append("contains_url")

    if has_email(q):
        reasons.append("contains_email")

    if contains_meta_keyword(q):
        reasons.append("meta_relation")

    if contains_technical_location(q):
        reasons.append("technical_location")

    # count-only — дропаем только если нет сильных compositional сигналов
    if starts_count(q) and fam == "count":
        if row["hop_count_estimate"] <= 1 and row["program_length"] <= 6:
            reasons.append("count_only_simple")

    # Узкие membership/date microfacts
    if contains_membership_microfact(q) and contains_date_microfact(q):
        reasons.append("membership_date_microfact")

    # Слишком искусственная вложенность
    if long_and_nested(q) and not list_style_friendly(q):
        reasons.append("too_nested_and_non_user_style")

    return reasons

def reconstructability_score(row: pd.Series) -> int:
    q = row["question_en_original"]
    fam = row["primary_reasoning_family"]
    tags = set(row["reasoning_tags"])
    sig = row["program_signature"]

    score = 0

    # Положительные сигналы
    if fam in PREFERRED_FAMILIES:
        score += 4
    elif fam in ALLOWED_BUT_LOWER_PRIORITY:
        score += 1

    if "multi_hop" in tags:
        score += 3
    if "logical_set" in tags:
        score += 3
    if "qualifier" in tags:
        score += 2
    if "temporal" in tags:
        score += 2
    if "comparison_or_superlative" in tags:
        score += 2

    score += min(row["hop_count_estimate"], 4)
    score += min(max(row["program_length"] - 4, 0), 4)

    if list_style_friendly(q):
        score += 3

    if program_has_set_ops(sig):
        score += 2

    # Отрицательные сигналы
    if starts_count(q):
        score -= 3
    if starts_binary(q):
        score -= 5
    if contains_meta_keyword(q):
        score -= 4
    if contains_technical_location(q):
        score -= 3
    if has_url(q) or has_email(q):
        score -= 5
    if contains_membership_microfact(q) and contains_date_microfact(q):
        score -= 3
    if long_and_nested(q):
        score -= 2

    # Короткие почти one-hop-подобные штуки понижаем
    if row["hop_count_estimate"] <= 1 and row["program_length"] <= 5:
        score -= 2

    return int(score)

def recommend_action(row: pd.Series) -> str:
    if len(row["drop_reasons"]) > 0:
        return "drop"
    score = row["reconstructability_score_v2"]
    fam = row["primary_reasoning_family"]

    if fam in PREFERRED_FAMILIES and score >= KEEP_FOR_RECONSTRUCTION_THRESHOLD:
        return "keep_for_reconstruction"

    if score >= PATTERN_ONLY_THRESHOLD:
        return "pattern_only"

    return "drop"


In [8]:
# =========================
# ПРИМЕНЕНИЕ ФИЛЬТРОВ
# =========================

df["drop_reasons"] = df.apply(hard_drop_reasons, axis=1)
df["num_drop_reasons"] = df["drop_reasons"].apply(len)
df["hard_keep"] = df["num_drop_reasons"] == 0

df["reconstructability_score_v2"] = df.apply(reconstructability_score, axis=1)
df["recommended_action"] = df.apply(recommend_action, axis=1)

df["question_len"] = df["question_en_original"].apply(len)
df["num_parentheses"] = df["question_en_original"].apply(parenthesis_count)
df["list_style_friendly"] = df["question_en_original"].apply(list_style_friendly)

action_df = (
    df.groupby("recommended_action")
    .agg(
        count=("benchmark_id", "count"),
        mean_score=("reconstructability_score_v2", "mean"),
        mean_program_length=("program_length", "mean"),
    )
    .reset_index()
    .sort_values("count", ascending=False)
)

action_df


,recommended_action,count,mean_score,mean_program_length
1,keep_for_reconstruction,38706,15.244019,5.894254
0,drop,29989,1.714695,5.142419
2,pattern_only,4096,6.676758,4.519775


In [9]:
# Какие причины дропа самые частые
drop_reason_counter = Counter()
for reasons in df["drop_reasons"]:
    for r in reasons:
        drop_reason_counter[r] += 1

drop_reasons_df = pd.DataFrame(
    [{"drop_reason": k, "count": v} for k, v in drop_reason_counter.most_common()]
)

drop_reasons_df.head(30)


,drop_reason,count
0,binary_question,15424
1,meta_relation,14147
2,family=verification,9674
3,contains_url,6375
4,count_only_simple,3666
5,technical_location,1167
6,too_nested_and_non_user_style,1160
7,membership_date_microfact,624
8,contains_email,5


In [10]:
# Смотрим family-статистику уже после новых фильтров/score

family_after_df = (
    df.groupby(["recommended_action", "primary_reasoning_family"])
    .agg(
        count=("benchmark_id", "count"),
        mean_score=("reconstructability_score_v2", "mean"),
        mean_program_length=("program_length", "mean"),
        mean_hops=("hop_count_estimate", "mean"),
    )
    .reset_index()
    .sort_values(["recommended_action", "count", "mean_score"], ascending=[True, False, False])
)

family_after_df.head(50)


,recommended_action,primary_reasoning_family,count,mean_score,mean_program_length,mean_hops
7,drop,verification,9674,-4.211908,4.716560,0.821687
4,drop,multi_hop,5953,11.803292,7.293801,1.313287
1,drop,count,4366,-0.096656,4.496564,1.000000
5,drop,qualifier,4174,1.645184,4.450168,0.957834
0,drop,comparison_or_superlative,3873,1.278079,3.459334,0.385231
3,drop,logical_set,1056,8.072917,8.067235,1.000000
2,drop,filtering,518,-1.359073,5.617761,1.000000
6,drop,temporal,375,7.165333,5.690667,1.000000
10,keep_for_reconstruction,multi_hop,23094,18.381571,6.963237,1.363211
8,keep_for_reconstruction,comparison_or_superlative,8284,9.781869,3.631217,0.565427


## Формируем pattern bank

Теперь мы уже не хотим тащить сотни похожих вопросов.  
Наша цель — оставить **разные полезные сигнатуры** и взять по несколько лучших примеров из каждой.


In [11]:
# =========================
# PATTERN BANK
# =========================

keep_df = df[df["recommended_action"].isin(["keep_for_reconstruction", "pattern_only"])].copy()
keep_df = keep_df.sort_values(
    ["recommended_action", "reconstructability_score_v2", "program_length", "question_len"],
    ascending=[True, False, False, False],
)

print("keep_df:", keep_df.shape)

signature_stats = (
    keep_df.groupby(["primary_reasoning_family", "program_signature"])
    .agg(
        count=("benchmark_id", "count"),
        best_score=("reconstructability_score_v2", "max"),
        mean_score=("reconstructability_score_v2", "mean"),
        mean_program_length=("program_length", "mean"),
        mean_hops=("hop_count_estimate", "mean"),
        example_question=("question_en_original", "first"),
    )
    .reset_index()
    .sort_values(["best_score", "count", "mean_program_length"], ascending=[False, False, False])
)

signature_stats.head(30)


keep_df: (42802, 34)


,primary_reasoning_family,program_signature,count,best_score,mean_score,mean_program_length,mean_hops,example_question
338,multi_hop,Find -> Relate -> Find -> And -> Relate -> Fil...,6,26,25.500000,10.0,3.0,"What person is a member, starting in 2012, of ..."
310,multi_hop,Find -> Relate -> Find -> And -> Relate -> Fil...,4,26,25.000000,12.0,3.0,Who is a cast member of Taken (which also has ...
387,multi_hop,Find -> Relate -> QFilterYear -> FilterConcept...,4,26,24.750000,12.0,3.0,"Which big city was, in 1958, a twinned adminis..."
361,multi_hop,Find -> Relate -> Find -> And -> Relate -> QFi...,2,26,26.000000,10.0,3.0,"Which agen was set up in a city that is, since..."
284,multi_hop,Find -> Relate -> Find -> And -> Find -> Relat...,772,25,21.611399,9.0,2.0,"Which is the longer of the two, From Here to E..."
283,multi_hop,Find -> Relate -> Find -> And -> Find -> Relat...,148,25,21.500000,9.0,2.0,What is the beginning date that Viacom (that i...
354,multi_hop,Find -> Relate -> Find -> And -> Relate -> QFi...,26,25,23.153846,8.0,2.0,"Where was TV series, comprising 151st part of ..."
240,multi_hop,Find -> Relate -> FilterConcept -> Find -> Rel...,19,25,24.368421,9.0,2.0,Which person received the 54th Annual Grammy A...
239,multi_hop,Find -> Relate -> FilterConcept -> Find -> Rel...,17,25,23.176471,9.0,2.0,What business has a member called the Green Ba...
378,multi_hop,Find -> Relate -> QFilterStr -> FilterConcept ...,16,25,23.125000,9.0,2.0,Which person received the Cannes Film Festival...


In [12]:
# Берём top-N примеров на сигнатуру
grouped = defaultdict(list)
for row in keep_df.to_dict("records"):
    key = (row["primary_reasoning_family"], row["program_signature"])
    grouped[key].append(row)

for key in grouped:
    grouped[key].sort(
        key=lambda r: (
            0 if r["recommended_action"] == "keep_for_reconstruction" else 1,
            -r["reconstructability_score_v2"],
            -r["program_length"],
            -r["hop_count_estimate"],
            -len(r["question_en_original"]),
            r["benchmark_id"],
        )
    )

pattern_bank = []
for key, rows_g in grouped.items():
    pattern_bank.extend(rows_g[:MAX_PER_SIGNATURE])

pattern_bank_df = pd.DataFrame(pattern_bank)
print("pattern_bank:", pattern_bank_df.shape)
pattern_bank_df.head(10)


pattern_bank: (1905, 34)


,benchmark_id,source_dataset,source_split,import_mode,role,question_ru,question_en_original,domain,reasoning_tags,primary_reasoning_family,...,reconstruction_prompt,notes,drop_reasons,num_drop_reasons,hard_keep,reconstructability_score_v2,recommended_action,question_len,num_parentheses,list_style_friendly
0,kqapro_train_013987,kqa_pro,train,template_transfer,template_transfer_candidate,None,"Which big city was, in 1958, a twinned adminis...",,"[filtering, high_compositional_depth, logical_...",multi_hop,...,Сгенерируй естественный запрос на русском язык...,,[],0,True,26,keep_for_reconstruction,212,2,True
1,kqapro_train_088566,kqa_pro,train,template_transfer,template_transfer_candidate,None,What musical received Grammy Award for Best Mu...,,"[filtering, high_compositional_depth, logical_...",multi_hop,...,Сгенерируй естественный запрос на русском язык...,,[],0,True,26,keep_for_reconstruction,149,2,True
2,kqapro_train_043170,kqa_pro,train,template_transfer,template_transfer_candidate,None,What Screen Actors Guild Award was nominated t...,,"[filtering, high_compositional_depth, logical_...",multi_hop,...,Сгенерируй естественный запрос на русском язык...,,[],0,True,24,keep_for_reconstruction,204,4,True
3,kqapro_train_055066,kqa_pro,train,template_transfer,template_transfer_candidate,None,For the country that is the Austrian Empire (t...,,"[filtering, high_compositional_depth, logical_...",multi_hop,...,Сгенерируй естественный запрос на русском язык...,,[],0,True,23,keep_for_reconstruction,192,3,False
4,kqapro_train_035873,kqa_pro,train,template_transfer,template_transfer_candidate,None,Who is a cast member of Taken (which also has ...,,"[filtering, high_compositional_depth, logical_...",multi_hop,...,Сгенерируй естественный запрос на русском язык...,,[],0,True,26,keep_for_reconstruction,149,2,True
5,kqapro_train_020084,kqa_pro,train,template_transfer,template_transfer_candidate,None,Which literary award was nominated to Graham G...,,"[filtering, high_compositional_depth, logical_...",multi_hop,...,Сгенерируй естественный запрос на русском язык...,,[],0,True,26,keep_for_reconstruction,134,2,True
6,kqapro_val_010512,kqa_pro,val,template_transfer,template_transfer_candidate,None,Which United States city is an administrative ...,,"[filtering, high_compositional_depth, logical_...",multi_hop,...,Сгенерируй естественный запрос на русском язык...,,[],0,True,24,keep_for_reconstruction,192,4,True
7,kqapro_train_030438,kqa_pro,train,template_transfer,template_transfer_candidate,None,What award was nominated to Bryan Adams (the o...,,"[filtering, high_compositional_depth, logical_...",multi_hop,...,Сгенерируй естественный запрос на русском язык...,,[],0,True,24,keep_for_reconstruction,163,4,True
8,kqapro_train_075279,kqa_pro,train,template_transfer,template_transfer_candidate,None,"What person is a member, starting in 2012, of ...",,"[filtering, high_compositional_depth, logical_...",multi_hop,...,Сгенерируй естественный запрос на русском язык...,,[],0,True,26,keep_for_reconstruction,167,2,True
9,kqapro_train_035339,kqa_pro,train,template_transfer,template_transfer_candidate,None,"Which human, in 2011, was nominated for the Go...",,"[filtering, high_compositional_depth, logical_...",multi_hop,...,Сгенерируй естественный запрос на русском язык...,,[],0,True,26,keep_for_reconstruction,160,2,True


## Балансируем итоговый shortlist

Здесь мы хотим получить финальный файл для ручного просмотра:
- не слишком большой;
- не заваленный одним семейством reasoning;
- с разнообразными `program_signature`.


In [13]:
# =========================
# ФИНАЛЬНЫЙ SHORTLIST
# =========================

family_buckets = defaultdict(list)
for row in pattern_bank_df.to_dict("records"):
    family_buckets[row["primary_reasoning_family"]].append(row)

for family in family_buckets:
    family_buckets[family].sort(
        key=lambda r: (
            0 if r["recommended_action"] == "keep_for_reconstruction" else 1,
            -r["reconstructability_score_v2"],
            -r["program_length"],
            -r["hop_count_estimate"],
        )
    )

preferred_family_order = [
    "multi_hop",
    "logical_set",
    "qualifier",
    "temporal",
    "comparison_or_superlative",
    "count",
]

final_shortlist = []
used_ids = set()

for family in preferred_family_order:
    rows_f = family_buckets.get(family, [])
    added = 0
    for row in rows_f:
        if row["benchmark_id"] in used_ids:
            continue
        final_shortlist.append(row)
        used_ids.add(row["benchmark_id"])
        added += 1
        if added >= MAX_PER_FAMILY:
            break

# Если до целевого объёма не добрали — добиваем лучшими остатками
if len(final_shortlist) < FINAL_SHORTLIST_TARGET:
    leftovers = []
    for family in preferred_family_order:
        for row in family_buckets.get(family, []):
            if row["benchmark_id"] not in used_ids:
                leftovers.append(row)
    leftovers.sort(
        key=lambda r: (
            0 if r["recommended_action"] == "keep_for_reconstruction" else 1,
            -r["reconstructability_score_v2"],
            -r["program_length"],
            -r["hop_count_estimate"],
        )
    )
    for row in leftovers:
        if len(final_shortlist) >= FINAL_SHORTLIST_TARGET:
            break
        final_shortlist.append(row)
        used_ids.add(row["benchmark_id"])

shortlist_df = pd.DataFrame(final_shortlist)
print("final_shortlist:", shortlist_df.shape)


final_shortlist: (180, 34)


In [14]:
shortlist_family_df = (
    shortlist_df.groupby(["primary_reasoning_family", "recommended_action"])
    .agg(
        count=("benchmark_id", "count"),
        mean_score=("reconstructability_score_v2", "mean"),
        mean_program_length=("program_length", "mean"),
        mean_hops=("hop_count_estimate", "mean"),
    )
    .reset_index()
    .sort_values(["primary_reasoning_family", "count"], ascending=[True, False])
)

shortlist_family_df


,primary_reasoning_family,recommended_action,count,mean_score,mean_program_length,mean_hops
0,comparison_or_superlative,keep_for_reconstruction,30,13.000000,5.000000,1.000000
1,count,pattern_only,30,10.600000,6.600000,1.000000
2,logical_set,keep_for_reconstruction,30,20.933333,9.000000,1.000000
3,multi_hop,keep_for_reconstruction,30,25.333333,10.400000,2.333333
4,qualifier,keep_for_reconstruction,30,17.000000,7.000000,1.000000
5,temporal,keep_for_reconstruction,30,12.400000,5.466667,1.000000


In [15]:
preview_cols = [
    "benchmark_id",
    "primary_reasoning_family",
    "recommended_action",
    "reconstructability_score_v2",
    "program_length",
    "hop_count_estimate",
    "reasoning_tags",
    "question_en_original",
    "answer_text",
]

shortlist_df[preview_cols].head(30)


,benchmark_id,primary_reasoning_family,recommended_action,reconstructability_score_v2,program_length,hop_count_estimate,reasoning_tags,question_en_original,answer_text
0,kqapro_train_013987,multi_hop,keep_for_reconstruction,26,12,3,"[filtering, high_compositional_depth, logical_...","Which big city was, in 1958, a twinned adminis...",Marseille
1,kqapro_train_088566,multi_hop,keep_for_reconstruction,26,12,3,"[filtering, high_compositional_depth, logical_...",What musical received Grammy Award for Best Mu...,Chicago
2,kqapro_train_035873,multi_hop,keep_for_reconstruction,26,12,3,"[filtering, high_compositional_depth, logical_...",Who is a cast member of Taken (which also has ...,Michael Jeter
3,kqapro_train_020084,multi_hop,keep_for_reconstruction,26,12,3,"[filtering, high_compositional_depth, logical_...",Which literary award was nominated to Graham G...,Neustadt International Prize for Literature
4,kqapro_train_075279,multi_hop,keep_for_reconstruction,26,10,3,"[filtering, high_compositional_depth, logical_...","What person is a member, starting in 2012, of ...",Thiago Silva
5,kqapro_train_035339,multi_hop,keep_for_reconstruction,26,10,3,"[filtering, high_compositional_depth, logical_...","Which human, in 2011, was nominated for the Go...",Robert Pattinson
6,kqapro_train_092631,multi_hop,keep_for_reconstruction,26,10,3,"[filtering, high_compositional_depth, logical_...",Who is the person who won MTV Video Music Awar...,Bruno Mars
7,kqapro_train_079707,multi_hop,keep_for_reconstruction,26,10,3,"[filtering, high_compositional_depth, logical_...",What is the award received by the relative of ...,Royal Victorian Chain
8,kqapro_train_046149,multi_hop,keep_for_reconstruction,26,10,3,"[filtering, high_compositional_depth, logical_...","Which agen was set up in a city that is, since...",Hudson Soft
9,kqapro_val_001197,multi_hop,keep_for_reconstruction,26,10,3,"[filtering, high_compositional_depth, logical_...",What disease is the medical condition of the p...,smallpox


In [16]:
# =========================
# MANUAL REVIEW SHEET
# =========================

manual_review_df = shortlist_df.copy()

manual_review_df["manual_decision"] = ""
manual_review_df["manual_bucket"] = ""   # keep_for_reconstruction / pattern_only / drop
manual_review_df["russian_reconstruction_idea"] = ""
manual_review_df["target_domain_ru"] = ""
manual_review_df["notes"] = ""

review_cols = [
    "benchmark_id",
    "primary_reasoning_family",
    "recommended_action",
    "reconstructability_score_v2",
    "program_signature",
    "program_length",
    "hop_count_estimate",
    "question_en_original",
    "answer_text",
    "manual_decision",
    "manual_bucket",
    "russian_reconstruction_idea",
    "target_domain_ru",
    "notes",
]

manual_review_df = manual_review_df[review_cols]
manual_review_df.head(10)


,benchmark_id,primary_reasoning_family,recommended_action,reconstructability_score_v2,program_signature,program_length,hop_count_estimate,question_en_original,answer_text,manual_decision,manual_bucket,russian_reconstruction_idea,target_domain_ru,notes
0,kqapro_train_013987,multi_hop,keep_for_reconstruction,26,Find -> Relate -> QFilterYear -> FilterConcept...,12,3,"Which big city was, in 1958, a twinned adminis...",Marseille,,,,,
1,kqapro_train_088566,multi_hop,keep_for_reconstruction,26,Find -> Relate -> QFilterYear -> FilterConcept...,12,3,What musical received Grammy Award for Best Mu...,Chicago,,,,,
2,kqapro_train_035873,multi_hop,keep_for_reconstruction,26,Find -> Relate -> Find -> And -> Relate -> Fil...,12,3,Who is a cast member of Taken (which also has ...,Michael Jeter,,,,,
3,kqapro_train_020084,multi_hop,keep_for_reconstruction,26,Find -> Relate -> Find -> And -> Relate -> Fil...,12,3,Which literary award was nominated to Graham G...,Neustadt International Prize for Literature,,,,,
4,kqapro_train_075279,multi_hop,keep_for_reconstruction,26,Find -> Relate -> Find -> And -> Relate -> Fil...,10,3,"What person is a member, starting in 2012, of ...",Thiago Silva,,,,,
5,kqapro_train_035339,multi_hop,keep_for_reconstruction,26,Find -> Relate -> Find -> And -> Relate -> Fil...,10,3,"Which human, in 2011, was nominated for the Go...",Robert Pattinson,,,,,
6,kqapro_train_092631,multi_hop,keep_for_reconstruction,26,Find -> Relate -> Find -> And -> Relate -> Fil...,10,3,Who is the person who won MTV Video Music Awar...,Bruno Mars,,,,,
7,kqapro_train_079707,multi_hop,keep_for_reconstruction,26,Find -> Relate -> Find -> And -> Relate -> Fil...,10,3,What is the award received by the relative of ...,Royal Victorian Chain,,,,,
8,kqapro_train_046149,multi_hop,keep_for_reconstruction,26,Find -> Relate -> Find -> And -> Relate -> QFi...,10,3,"Which agen was set up in a city that is, since...",Hudson Soft,,,,,
9,kqapro_val_001197,multi_hop,keep_for_reconstruction,26,Find -> Relate -> Find -> And -> Relate -> QFi...,10,3,What disease is the medical condition of the p...,smallpox,,,,,


In [17]:
# =========================
# ЭКСПОРТ
# =========================

filtered_rows = keep_df.to_dict("records")
pattern_bank_rows = pattern_bank_df.to_dict("records")
shortlist_rows = shortlist_df.to_dict("records")

save_jsonl(JSONL_DIR / "kqapro_filtered_candidates.jsonl", filtered_rows)
save_jsonl(JSONL_DIR / "kqapro_pattern_bank.jsonl", pattern_bank_rows)
save_jsonl(JSONL_DIR / "kqapro_reconstruction_shortlist.jsonl", shortlist_rows)

keep_df.to_csv(CSV_DIR / "kqapro_filtered_candidates.csv", index=False)
pattern_bank_df.to_csv(CSV_DIR / "kqapro_pattern_bank.csv", index=False)
shortlist_df.to_csv(CSV_DIR / "kqapro_reconstruction_shortlist.csv", index=False)

action_df.to_csv(CSV_DIR / "kqapro_action_summary.csv", index=False)
drop_reasons_df.to_csv(CSV_DIR / "kqapro_drop_reasons_summary.csv", index=False)
family_after_df.to_csv(CSV_DIR / "kqapro_family_summary_after_filters.csv", index=False)
signature_stats.to_csv(CSV_DIR / "kqapro_signature_stats.csv", index=False)
shortlist_family_df.to_csv(CSV_DIR / "kqapro_shortlist_family_summary.csv", index=False)
manual_review_df.to_csv(CSV_DIR / "kqapro_manual_review_sheet.csv", index=False)

summary = {
    "input_path": str(INPUT_PATH),
    "num_input_rows": int(len(df)),
    "num_keep_candidates": int(len(keep_df)),
    "num_pattern_bank_rows": int(len(pattern_bank_df)),
    "num_final_shortlist_rows": int(len(shortlist_df)),
    "keep_threshold": int(KEEP_FOR_RECONSTRUCTION_THRESHOLD),
    "pattern_only_threshold": int(PATTERN_ONLY_THRESHOLD),
    "max_per_signature": int(MAX_PER_SIGNATURE),
    "max_per_family": int(MAX_PER_FAMILY),
    "final_shortlist_target": int(FINAL_SHORTLIST_TARGET),
    "preferred_families": sorted(PREFERRED_FAMILIES),
    "drop_families": sorted(DROP_FAMILIES),
}
with open(REPORTS_DIR / "kqapro_stage3_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("saved JSONL to:", JSONL_DIR)
print("saved CSV   to:", CSV_DIR)
print("saved report to:", REPORTS_DIR / "kqapro_stage3_summary.json")
print("\nKey files:")
print("-", JSONL_DIR / "kqapro_filtered_candidates.jsonl")
print("-", JSONL_DIR / "kqapro_pattern_bank.jsonl")
print("-", JSONL_DIR / "kqapro_reconstruction_shortlist.jsonl")
print("-", CSV_DIR / "kqapro_manual_review_sheet.csv")


saved JSONL to: /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage3/jsonl
saved CSV   to: /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage3/csv
saved report to: /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage3/reports/kqapro_stage3_summary.json

Key files:
- /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage3/jsonl/kqapro_filtered_candidates.jsonl
- /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage3/jsonl/kqapro_pattern_bank.jsonl
- /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage3/jsonl/kqapro_reconstruction_shortlist.jsonl
- /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage3/csv/kqapro_manual_review_sheet.csv


## Как читать выход

Самые важные файлы после запуска:

- `artifacts_kqapro_stage3/jsonl/kqapro_filtered_candidates.jsonl`  
  Всё, что прошло жёсткие фильтры и ещё может быть полезно.

- `artifacts_kqapro_stage3/jsonl/kqapro_pattern_bank.jsonl`  
  Уже **сжатый банк паттернов**: по несколько лучших примеров на сигнатуру.

- `artifacts_kqapro_stage3/jsonl/kqapro_reconstruction_shortlist.jsonl`  
  Финальный shortlist для ручного просмотра и русской реконструкции.

- `artifacts_kqapro_stage3/csv/kqapro_manual_review_sheet.csv`  
  Таблица, где можно руками отмечать:
  - оставить ли паттерн,
  - как пересобрать его по-русски,
  - в какой домен твоего benchmark-а он ляжет.
